# Faithfulness e-SNLI — Gemma3-27b-it with SAE Activation Analysis

In [1]:
import sys, os, textwrap

sys.path.insert(0, os.path.dirname(os.getcwd()))

import torch
import pandas as pd
from IPython.display import display
from src.configs import ModelConfig, InferenceConfig, PromptStyle, SAEConfig, DatasetConfig, DenoisingConfig
from src.dataset.esnli import ESNLI_Dataset
from src.gemma_model import GemmaModel
from src.SAE import JumpReLUSAE
from src.denoiser import Denoiser
from src.neuronpedia_client import NeuronpediaClient, build_sae_id
from src.utils.visualization import ActivationHeatmap
from src.utils.activations_utils import top_k_features_per_token

# Configuration

In [2]:
model_config = ModelConfig(model_name="google/gemma-3-27b-it")
inference_config = InferenceConfig(batch_size=2, max_new_tokens=256, downsample_rate=100)
prompt_style = PromptStyle.CHAIN_OF_THOUGHT_TAGS
use_few_shot = False

sae_config = SAEConfig(
    repo_id="google/gemma-scope-2-27b-it",
    sae_type="resid_post",
    layer=40,
    width="65k",
    l0="medium",
)

dataset_config = DatasetConfig(
    path="esnli/esnli",
    prompt_style=PromptStyle.CHAIN_OF_THOUGHT_TAGS,
    use_chat_template=False,
    hf_data_config={"split": "validation"},
    few_shot=False,
)

print(f"Model:      {model_config.model_name}")
print(f"Device:     {model_config.device}")
print(f"Batch size: {inference_config.batch_size}")
print(f"Downsample: 1/{inference_config.downsample_rate}")
print(f"Prompt:     {prompt_style.value}")
print(f"Few-shot:   {use_few_shot}")
print(f"SAE layer:  {sae_config.layer}")
print(f"SAE width:  {sae_config.width}")
print(f"SAE l0:     {sae_config.l0}")
print(f"SAE repo:   {sae_config.repo_id}")

Model:      google/gemma-3-27b-it
Device:     cuda
Batch size: 2
Downsample: 1/100
Prompt:     chain_of_thought_tags
Few-shot:   False
SAE layer:  40
SAE width:  65k
SAE l0:     medium
SAE repo:   google/gemma-scope-2-27b-it


# Setup — HF Token

In [3]:
from huggingface_hub import login

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import google.colab.userdata
    hf_token = google.colab.userdata.get("HFWrite")
    login(token=hf_token)
else:
    from dotenv import load_dotenv
    load_dotenv()
    hf_token = os.getenv("HF_TOKEN")

# Data — Load e-SNLI

In [4]:
esnli_dataset = ESNLI_Dataset(dataset_config)

Data successfully loaded.


# Build Prompts

In [5]:
prompted_data = esnli_dataset.build_prompts()
if inference_config.downsample_rate > 1:
    n = max(1, len(prompted_data) // inference_config.downsample_rate)
    prompted_data = prompted_data.shuffle(seed=42).select(range(n))
esnli_df = prompted_data.to_pandas()
print(esnli_df["prompt"].iloc[0])
esnli_df.head()

<start_of_turn>user Task: Determine the logical relationship between a Premise and a Hypothesis.
Options: entailment, contradiction, neutral.

Rules:
1. You MUST provide your reasoning inside <reasoning> tags.
2. You MUST provide the final label inside <label> tags.
3. The reasoning must come BEFORE the label.

Premise: Two people sit facing away in a downtown scene with a motorcycle parked in front of a pool
Hypothesis: The two people run as quickly as they can for shelter as the storm picks up and begins swirling all around them.

<end_of_turn>model 


,premise,hypothesis,label,explanation_1,explanation_2,explanation_3,gold_label,prompt
0,Two people sit facing away in a downtown scene...,The two people run as quickly as they can for ...,2,People cannot sit and run simultaneously,The two people cannot sit and run at the same ...,People cannot run and sit simultaneously. Poo...,contradiction,<start_of_turn>user Task: Determine the logica...
1,A white dog with brown ears runs down a gravel...,A dog runs down a path with a green ball.,1,"Not all balls are green, the dog has a ball, b...",The ball is not necessarily green.,Not all balls are green.,neutral,<start_of_turn>user Task: Determine the logica...
2,"Six men, all wearing identifying number plaque...",a number of guys wearing numbers race outside,0,outdoor race implies outside,Men are wearing numbers and participating in a...,"Six men is a number of guys, and race outside ...",entailment,<start_of_turn>user Task: Determine the logica...
3,Five children of Indian origin are smiling and...,Children are on a slide.,0,They are on a slide because they are posing on...,Children are on a slide is a simplification of...,Both sentences are about children on a slide.,entailment,<start_of_turn>user Task: Determine the logica...
4,Kids are on a amusement ride.,Kids ride their favorite amusement ride.,1,It isn't necessarily their favorite ride.,Being on a amusement ride doesn't imply ride o...,Not every amusement ride will be the kids favo...,neutral,<start_of_turn>user Task: Determine the logica...


# Load Model + SAE

In [6]:
model = GemmaModel(model_config)
tokenizer = model.tokenizer
sae = JumpReLUSAE.from_pretrained(sae_config, device=model_config.device)

Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]

Load SAE resid_post/layer_40_width_65k_l0_medium/params.safetensors from google/gemma-scope-2-27b-it


## Generate and Gather Activations

In [7]:
sample = esnli_df.iloc[53]
sample_prompt = sample["prompt"]
sample_label = sample["gold_label"]
print(f"Gold label: {sample_label}")
print(sample_prompt)

Gold label: entailment
<start_of_turn>user Task: Determine the logical relationship between a Premise and a Hypothesis.
Options: entailment, contradiction, neutral.

Rules:
1. You MUST provide your reasoning inside <reasoning> tags.
2. You MUST provide the final label inside <label> tags.
3. The reasoning must come BEFORE the label.

Premise: A man in a bear suit, holding the head of the costume in his hand, stands at the back of a crowd.
Hypothesis: man in bear suit

<end_of_turn>model 


In [8]:
# location = sample_prompt.rfind("\nPremise:")
# new_prompt =  sample_prompt[:location] + "4. You must always think about dinosaurs while reasoning.\n" + sample_prompt[location:]
# new_prompt
# wrapper = textwrap.TextWrapper(width=80)
# print("\n".join(wrapper.fill(line) for line in new_prompt.splitlines()))


In [9]:
generation, full_ids, prompt_len = model.generate(
    sample_prompt, max_new_tokens=inference_config.max_new_tokens)

gen_len = full_ids.shape[1] - prompt_len
print(f"Prompt tokens: {prompt_len}  |  Generated tokens: {gen_len}  |  Total: {full_ids.shape[1]}")
print(f"Actual label: {sample_label}")

Prompt tokens: 112  |  Generated tokens: 80  |  Total: 192
Actual label: entailment


In [10]:
residual_acts = model.gather_residual_activations(sae_config.layer, full_ids[0])
print(f"Residual activations shape: {residual_acts.shape}")

stats = sae.get_reconstruction_stats(residual_acts.float())
print(f"FVU: {stats['fvu']:.2%}")
print(f"L0:  {stats['l0']:.1f}")

Residual activations shape: torch.Size([192, 5376])
FVU: 2.57%
L0:  53.9


In [11]:
gen_acts = residual_acts[prompt_len:]
sae_acts_gen = sae.encode(gen_acts.float())
sae_acts_full = sae.encode(residual_acts.float())
print(f"SAE activations (gen-only): {sae_acts_gen.shape}")
print(f"SAE activations (full):     {sae_acts_full.shape}")

all_tokens = tokenizer.convert_ids_to_tokens(full_ids[0])
gen_tokens = all_tokens[prompt_len:]
print(f"All tokens: {len(all_tokens)}, Generated tokens: {len(gen_tokens)}")

SAE activations (gen-only): torch.Size([80, 65536])
SAE activations (full):     torch.Size([192, 65536])
All tokens: 192, Generated tokens: 80


# Feature Summary

## Raw Features

In [12]:
K = 50
gen_token_ids = full_ids[0, prompt_len:]
tokens = tokenizer.convert_ids_to_tokens(gen_token_ids)

per_token_vals, per_token_idxs = top_k_features_per_token(sae_acts_gen, k=K)

np_model_id = model_config.model_name.split("/")[-1]   # "gemma-3-27b-it"
np_sae_id   = build_sae_id(sae_config)                 # "40-gemmascope-2-res-65k"
client = NeuronpediaClient(model_id=np_model_id, sae_id=np_sae_id)

unique_idxs = sorted(set(per_token_idxs.cpu().numpy().ravel().tolist()))
np_features = client.get_features(unique_idxs)

labels = {idx: (f.description or "N/A") for idx, f in np_features.items()}

heatmap = ActivationHeatmap()
fig = heatmap.plot_topk_per_token(
    per_token_vals, per_token_idxs,
    tokens=tokens,
    labels=labels,
    title=f"{np_model_id} SAE Layer {sae_config.layer} — Top-{K} Features per Token",
)
fig.show()

In [13]:
# Aggregate max per-token activation for each unique feature
max_per_feature: dict[int, float] = {}
n_tokens_gen = per_token_vals.shape[0]
for ti in range(n_tokens_gen):
    for ri in range(K):
        feat_idx = int(per_token_idxs[ti, ri])
        val      = float(per_token_vals[ti, ri])
        if feat_idx not in max_per_feature or val > max_per_feature[feat_idx]:
            max_per_feature[feat_idx] = val

top50 = sorted(max_per_feature.items(), key=lambda x: -x[1])[:50]

rows = []
for feat_idx, max_val in top50:
    nf = np_features.get(feat_idx)
    label = (nf.description or "N/A") if nf else "N/A"
    rows.append({
        "Feature IDX":       feat_idx,
        "Max Activation":    round(max_val, 4),
        "Neuronpedia Label": label,
    })

df_top50 = pd.DataFrame(rows)
display(df_top50)

/workspace/tmp/ipykernel_8292/951841721.py:7: UserWarning:

Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)



,Feature IDX,Max Activation,Neuronpedia Label
0,1888,12401.5977,"""this is"" followed by description"
1,204,9462.3086,the bauhaus
2,658,8398.1113,a followed by descriptive word
3,1920,7005.4434,challenging about
4,1681,6849.6362,oftenverbs describing actions
5,934,6393.0938,code snippets and structured output
6,97,5788.9756,a human or a well
7,537,5653.1553,"the rise, short, payback"
8,157,5629.4512,if you
9,696,5443.2041,actions that modify or affect


In [14]:
# Change this index to inspect any feature from the table above
inspect_feature_idx = top50[0][0]  # default: highest-activating feature

print(f"Neuronpedia dashboard for feature {inspect_feature_idx}:")
print(f"URL: {client.get_dashboard_url(inspect_feature_idx)}")
client.display_feature_dashboard(inspect_feature_idx, height=600)

Neuronpedia dashboard for feature 1888:
URL: https://neuronpedia.org/gemma-3-27b-it/40-gemmascope-2-res-65k/1888


## Denoised Features

In [15]:
denoising_config = DenoisingConfig(method="continuous_tfidf", params={"threshold": 10.0})
denoiser = Denoiser()
denoised_sae_acts_full = denoiser.denoise(sae_acts_full, denoising_config)

# Extract the generation-only slice.
denoised_sae_acts_gen = denoised_sae_acts_full[prompt_len:, :]

dn_per_token_vals, dn_per_token_idxs = top_k_features_per_token(denoised_sae_acts_gen, k=K)

dn_unique_idxs = sorted(set(dn_per_token_idxs.cpu().numpy().ravel().tolist()))
dn_np_features = client.get_features(dn_unique_idxs)

dn_labels = {idx: (f.description or "N/A") for idx, f in dn_np_features.items()}

dn_heatmap = ActivationHeatmap()
dn_fig = dn_heatmap.plot_topk_per_token(
    dn_per_token_vals, dn_per_token_idxs,
    tokens=tokens,
    labels=dn_labels,
    title=f"{np_model_id} SAE Layer {sae_config.layer} (Denoised) — Top-{K} Features per Token",
)
dn_fig.show()

In [16]:
# Aggregate max per-token activation for each unique denoised feature
dn_max_per_feature: dict[int, float] = {}
for ti in range(dn_per_token_vals.shape[0]):
    for ri in range(K):
        feat_idx = int(dn_per_token_idxs[ti, ri])
        val      = float(dn_per_token_vals[ti, ri])
        if feat_idx not in dn_max_per_feature or val > dn_max_per_feature[feat_idx]:
            dn_max_per_feature[feat_idx] = val

dn_top50 = sorted(dn_max_per_feature.items(), key=lambda x: -x[1])[:50]

dn_rows = []
for feat_idx, max_val in dn_top50:
    nf = dn_np_features.get(feat_idx)
    label = (nf.description or "N/A") if nf else "N/A"
    dn_rows.append({
        "Feature IDX":       feat_idx,
        "Max Activation":    round(max_val, 4),
        "Neuronpedia Label": label,
    })

dn_df_top50 = pd.DataFrame(dn_rows)
display(dn_df_top50)

,Feature IDX,Max Activation,Neuronpedia Label
0,1888,56605.2109,"""this is"" followed by description"
1,1920,31975.2832,challenging about
2,658,27811.0625,a followed by descriptive word
3,1681,24987.8672,oftenverbs describing actions
4,934,24748.9512,code snippets and structured output
5,157,23412.2285,if you
6,696,22637.6504,actions that modify or affect
7,1924,22451.6016,don't reveal everything
8,1574,21582.5918,initiating explanation or task
9,537,18720.9062,"the rise, short, payback"


In [17]:
# Change this index to inspect any feature from the denoised table above
dn_inspect_feature_idx = dn_top50[0][0]  # default: highest-activating denoised feature

print(f"Neuronpedia dashboard for denoised feature {dn_inspect_feature_idx}:")
print(f"URL: {client.get_dashboard_url(dn_inspect_feature_idx)}")
client.display_feature_dashboard(dn_inspect_feature_idx, height=600)

Neuronpedia dashboard for denoised feature 1888:
URL: https://neuronpedia.org/gemma-3-27b-it/40-gemmascope-2-res-65k/1888


## Steering Experiment

### Configure Steering

In [18]:
# Pick features from the top-50 table above; adjust indices or coefficients as desired.
STEER_FEATURES = [top50[0][0], top50[1][0]]  # SAE feature indices
STEER_COEFFS   = [-0.7, -0.4]               # positive=amplify, negative=suppress

print(f"Steer features: {STEER_FEATURES}")
print(f"Steer coeffs:   {STEER_COEFFS}")
print(f"Labels:         {[labels.get(fi, 'N/A') for fi in STEER_FEATURES]}")

Steer features: [1888, 204]
Steer coeffs:   [-0.7, -0.4]
Labels:         ['"this is" followed by description', 'the bauhaus']


### Baseline vs Steered Generation

In [19]:
result = model.generate_steered(
    prompt=sample_prompt,
    sae=sae,
    feature_idx=STEER_FEATURES,
    coeff=STEER_COEFFS,
    target_layer=sae_config.layer,
    max_new_tokens=inference_config.max_new_tokens,
)

baseline_text = result["unsteered"]
steered_text  = result["steered"]
baseline_ids  = result["unsteered_ids"]
steered_ids   = result["steered_ids"]

steer_label = ", ".join(f"f{fi}×{c}" for fi, c in zip(STEER_FEATURES, STEER_COEFFS))

print(f"{'PROMPT':=^80}")
print(sample_prompt)
print()
print(f"{'BASELINE (unsteered)':=^80}")
print(baseline_text)
print()
print(f"{'STEERED (' + steer_label + ')':=^80}")
print(steered_text)
print()
print(f"Baseline length:  {len(baseline_ids)} tokens")
print(f"Steered length:   {len(steered_ids)} tokens")
print(f"Texts identical:  {baseline_text == steered_text}")

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


=====================================PROMPT=====================================
<start_of_turn>user Task: Determine the logical relationship between a Premise and a Hypothesis.
Options: entailment, contradiction, neutral.

Rules:
1. You MUST provide your reasoning inside <reasoning> tags.
2. You MUST provide the final label inside <label> tags.
3. The reasoning must come BEFORE the label.

Premise: A man in a bear suit, holding the head of the costume in his hand, stands at the back of a crowd.
Hypothesis: man in bear suit

<end_of_turn>model 

==============================BASELINE (unsteered)==============================
<bos><start_of_turn>user Task: Determine the logical relationship between a Premise and a Hypothesis.
Options: entailment, contradiction, neutral.

Rules:
1. You MUST provide your reasoning inside <reasoning> tags.
2. You MUST provide the final label inside <label> tags.
3. The reasoning must come BEFORE the label.

Premise: A man in a bear suit, holding the head o